[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/125/blob/main/07_dataframes/07_4_Groups_and_Crosstabs.ipynb)

# 07.4: Groups and Crosstabs

Most interesting questions compare groups. Do musicals earn more than plays? Which theatre brought in the most money? You could answer these with a mask for each group, but that means writing one mask per group, and this data has 56 theatres. This notebook introduces `.groupby()`, which does the work for every group at once, and `pd.crosstab()`, which counts combinations of two categories.

This notebook opens in Google Colab. If Colab warns that the notebook was not authored by Google, click **Run anyway**. Your changes are not kept when you close the tab unless you choose **File > Save a copy in Drive**, so do that if you want to keep your work.

## The data

We use the Broadway data again, where each row is one show in one week.

In [ ]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_corgis_broadway.csv")
df.head(2)

## The problem with masks

Here is one comparison done with masks.

In [ ]:
print(df[df["Type"] == "Musical"]["Gross"].mean())
print(df[df["Type"] == "Play"]["Gross"].mean())
print(df[df["Type"] == "Special"]["Gross"].mean())

This works, but it repeats the same code for each group.

## `.groupby()`

`.groupby()` splits the rows into groups based on the values in a column, calculates something for each group, and combines the answers into one result. This pattern is called **split-apply-combine**. You write three things: the column that defines the groups, the column to calculate on, and the calculation.

In [ ]:
df.groupby("Type")["Gross"].mean()

These are the same three numbers as above. A musical takes in about 700312 in a typical week, a special about 365322 and a play about 325718. Read the code from left to right: "group by Type, look at Gross, and take the mean." The result is a series, where the index holds the groups, so you can use everything from module 06 on it. Any of the statistics you know will work here, including `.median()`, `.sum()`, `.min()`, `.max()` and `.count()`.

In [ ]:
df.groupby("Type")["Gross"].count()

`.count()` counts the values that are not missing in the column you chose. `.size()` counts all of the rows in each group, missing or not. They agree here because nothing is missing in `Gross`, but they would differ on a column with gaps.

In [ ]:
df.groupby("Type").size()

Always check how many rows are behind a group's average. Specials have a similar mean to plays, but that mean is based on only 339 rows compared with 8406 for plays.

## Sorting a grouped result

Since the result is a series, `.sort_values()` and `.head()` give you a top-N list. Which shows brought in the most money in total?

In [ ]:
df.groupby("Name")["Gross"].sum().sort_values(ascending=False).head(3)

The Lion King leads with about 1.29 billion dollars, followed by Wicked at about 1.04 billion and The Phantom Of The Opera at about 0.79 billion. Now compare that with a different column to group by. Which theatres brought in the most?

In [ ]:
df.groupby("Theatre")["Gross"].sum().sort_values(ascending=False).head(3)

The Gershwin, where Wicked plays, leads with about 1.22 billion. Each of these took one line, and they answer questions that would have needed a mask for each of 820 shows or 56 theatres.

You can also look at part of a grouped result, since it is an ordinary series or table. Which month has the highest mean weekly gross?

In [ ]:
df.groupby("Month")["Gross"].mean().sort_values(ascending=False).head(2)

Month 8, August, is highest at about 667351, followed by July at about 655001, so summer is strong for Broadway.

## Several statistics at once

`.agg()` calculates several statistics for one column, with one column in the result for each statistic you list.

In [ ]:
df.groupby("Type")["Gross"].agg(["count", "mean", "median", "max"])

This one table shows that musicals have the most weeks and the highest mean of about 700312, and that for all three types the mean is above the median, which points to a long tail of very high weeks.

## More than one column

To calculate on several columns, pass a list of column names. The result is a DataFrame with one row per group.

In [ ]:
df.groupby("Type")[["Gross", "Attendance"]].mean()

Musicals lead on both, with a mean attendance of about 9131 per week against about 5122 for plays. Notice that `Type` is in the index of this result. If you want it back as an ordinary column, which is what you will need for making charts in module 08, use `.reset_index()`.

In [ ]:
df.groupby("Type")["Gross"].mean().reset_index()

The result is now a small DataFrame with two ordinary columns, `Type` and `Gross`. In this course we group by one column at a time. When a question involves two categories at once, we use a crosstab.

## Crosstabs: counting two categories at once

`pd.crosstab()` counts how many rows fall into each combination of two categories. To have a second category to work with, we can make one with a mask: was this a week with a gross of more than a million dollars?

In [ ]:
df["Million_Week"] = df["Gross"] > 1000000
pd.crosstab(df["Type"], df["Million_Week"])

Each number counts rows. The first argument gives the rows of the table and the second gives the columns, so musicals had 3906 million-dollar weeks and plays had 175. Raw counts are hard to compare here because there are so many more musical rows than play rows. `normalize="index"` converts each row into fractions that add up to 1.

In [ ]:
pd.crosstab(df["Type"], df["Million_Week"], normalize="index")

Now the rows can be compared directly. About 17 percent of musical weeks reached a million dollars, compared with about 9 percent for specials and about 2 percent for plays. Be careful about what each fraction is a share of: `normalize="index"` gives the share within each row, so 0.173 is the fraction of musical weeks, and not the fraction of million-dollar weeks that were musicals. `margins=True` adds a row and a column of totals to a table of counts.

In [ ]:
pd.crosstab(df["Type"], df["Million_Week"], margins=True)

The `All` row and column give the totals: 4111 of the 31296 rows are million-dollar weeks.

`normalize="columns"` gives shares within each column instead, which answers a different question. It says what fraction of the million-dollar weeks came from each type of show.

In [ ]:
pd.crosstab(df["Type"], df["Million_Week"], normalize="columns").round(3)

In the `True` column, about 95 percent of million-dollar weeks were musicals, about 4 percent were plays and about 1 percent were specials. Compare that with the 17 percent from the earlier table, which used the same counts. The question decides which one you want: 17 percent of musical weeks reached a million dollars, while 95 percent of million-dollar weeks were musicals.

## Making categories from numbers: `pd.cut()`

A crosstab needs categories. `pd.cut()` makes categories from a numeric column. You give it the edges of the ranges in `bins` and a name for each range in `labels`. By default each range includes its upper edge and not its lower edge, so 1989 to 1999 covers the years 1990 through 1999.

In [ ]:
df["Period"] = pd.cut(df["Year"], bins=[1989, 1999, 2009, 2016], labels=["1990s", "2000s", "2010s"])
pd.crosstab(df["Period"], df["Type"], normalize="index")

Reading across each row gives the mix of show types in that period. Musicals grew from about 69 percent of show-weeks in the 1990s to about 74 percent in the 2010s, while plays fell from about 29 percent to about 26 percent. Note that the data stops in August of 2016, so the 2010s row covers fewer years than the others, but fractions make that less of a problem than counts would.

## Practice problems

These problems are not collected or graded. They are here so that you can find out whether you can write this code yourself before the reading check and the pair activity. Write your answer in the cell under each problem, run it, and then run the check cell below it, which tells you whether your answer is right. The check cells are collapsed because they contain the answers, so try the problem before you open one.

Run the next two cells first. The first one creates fresh data for the problems, and the second sets up the checks.

In [ ]:
import pandas as pd

majors = pd.read_csv("https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_recent_grads.csv")
majors["High_Pay"] = majors["Median"] >= 40000
majors_start = majors.copy()

In [ ]:
#@title Run this cell to set up the checks
import numpy as np
import pandas as pd


def same(got, want):
    """True when got matches want: same labels and values for a series, same value for a number."""
    try:
        if isinstance(want, pd.DataFrame):
            if not isinstance(got, pd.DataFrame) or list(got.columns) != list(want.columns) or len(got) != len(want):
                return False
            if not pd.api.types.is_integer_dtype(want.index) and list(got.index) != list(want.index):
                return False
            got, want = got.reset_index(drop=True), want.reset_index(drop=True)
            for column in want.columns:
                if pd.api.types.is_numeric_dtype(want[column]):
                    if not np.allclose(pd.to_numeric(got[column], errors="coerce").astype(float), want[column].astype(float), equal_nan=True):
                        return False
                elif got[column].fillna("").astype(str).to_list() != want[column].fillna("").astype(str).to_list():
                    return False
            return True
        if isinstance(want, pd.Series):
            if not isinstance(got, pd.Series) or list(got.index) != list(want.index):
                return False
            if pd.api.types.is_numeric_dtype(want):
                return bool(np.allclose(got.astype(float), want.astype(float), equal_nan=True))
            return list(got) == list(want)
        if isinstance(want, (list, np.ndarray)):
            return list(got) == list(want)
        if isinstance(want, str):
            return got == want
        if isinstance(got, (pd.Series, pd.DataFrame, list, np.ndarray)):
            return False
        return bool(np.isclose(float(got), float(want)))
    except (TypeError, ValueError):
        return False


def check(number, name, want, hint, unchanged=None):
    """Print feedback for one problem. unchanged is (variable name, its starting value)."""
    if name not in globals():
        print(f"Problem {number}: not yet. No variable named {name} exists. Did you run your answer cell?")
    elif not same(globals()[name], want):
        print(f"Problem {number}: not yet. {hint}")
    elif unchanged is not None and not globals()[unchanged[0]].equals(unchanged[1]):
        print(f"Problem {number}: {name} is right, but {unchanged[0]} changed. Work on a copy, then run the setup cell again.")
    else:
        print(f"Problem {number}: correct.")

### Problem 1
Create a series named `median_pay` that gives the mean of the `Median` column for each `Major_category`.

In [ ]:
# Problem 1

In [ ]:
#@title Check Problem 1
check(1, "median_pay", majors_start.groupby("Major_category")["Median"].mean(), "Group by Major_category, select Median, then call .mean().")

### Problem 2
Create a series named `majors_per_category` that gives the number of rows for each `Major_category`.

In [ ]:
# Problem 2

In [ ]:
#@title Check Problem 2
check(2, "majors_per_category", majors_start.groupby("Major_category").size(), "Call .size() or .count() after grouping.")

### Problem 3
Create a series named `top_employment` with the three categories that have the highest total `Employed`, ordered from highest to lowest.

In [ ]:
# Problem 3

In [ ]:
#@title Check Problem 3
check(3, "top_employment", majors_start.groupby("Major_category")["Employed"].sum().sort_values(ascending=False).head(3), "Group, take the sum of Employed, sort from high to low and take the first three.")

### Problem 4
Create a DataFrame named `pay_table` that has `Major_category` and `Median` as ordinary columns, where `Median` is the mean for each category.

In [ ]:
# Problem 4

In [ ]:
#@title Check Problem 4
check(4, "pay_table", majors_start.groupby("Major_category")["Median"].mean().reset_index(), "Add .reset_index() to the grouped result.")

### Problem 5
Create a DataFrame named `pay_stats` with the `min`, `mean` and `max` of `Median` for each category, in that order.

In [ ]:
# Problem 5

In [ ]:
#@title Check Problem 5
check(5, "pay_stats", majors_start.groupby("Major_category")["Median"].agg(["min", "mean", "max"]), "Use .agg() with a list of the three names.")

### Problem 6
`High_Pay` is `True` for majors with a median of at least 40000. Create a crosstab named `pay_counts` with `Major_category` as the rows and `High_Pay` as the columns.

In [ ]:
# Problem 6

In [ ]:
#@title Check Problem 6
check(6, "pay_counts", pd.crosstab(majors_start["Major_category"], majors_start["High_Pay"]), "Pass the two columns to pd.crosstab(), rows first.")

### Problem 7
Create the same crosstab, named `pay_shares`, so that each row adds up to 1.

In [ ]:
# Problem 7

In [ ]:
#@title Check Problem 7
check(7, "pay_shares", pd.crosstab(majors_start["Major_category"], majors_start["High_Pay"], normalize="index"), "Think about which value of normalize= gives shares within each row.")

### Problem 8
Create a series named `pay_band` that uses `pd.cut()` to label each value of `Median` as `low` for up to 30000, `middle` for above 30000 up to 50000, and `high` for above 50000. Use bins of 0, 30000, 50000 and 200000.

In [ ]:
# Problem 8

In [ ]:
#@title Check Problem 8
check(8, "pay_band", pd.cut(majors_start["Median"], bins=[0, 30000, 50000, 200000], labels=["low", "middle", "high"]).astype(str), "Give pd.cut() the column, bins= and labels=.")

## What's next

That finishes module 07. You can now load a dataset, check it, select from it and compare groups. All of your results so far have been tables of numbers, though, and some patterns are much easier to see in a picture. Module 08 starts with notebook 08.1, which shows how to make charts.